## **HuggingFace transformers 的 AutoTokenizer 类与模型词汇表说明**

&emsp;&emsp;对用户请求生成 token 的过程对大模型而言可能通过不同分词器实现，而 vLLM 使用 HuggingFace 的 transformers 模块中 AutoTokenizer 类实现用户请求的词元化（tokenized）与 token ID 映射的过程。

**tokenizer 分词器来源与词汇表的关系：**

- 分词器模型发布方提供：
  - 词汇表（token → ID 的映射）
  - 分词规则（BPE/SentencePiece 合并规则）
  - 参考实现（官方配套的分词器具有已训练完成的词汇表与分词规则）
- 第三方分词器可以使用，但必须：
  - 加载相同的词汇表
  - 遵循相同的分词规则
  - 产生相同的 token ID 序列
- 否则：token ID 不匹配 → 模型输入错误 → 输出乱码
- **第三方分词器本身不需要词汇表和分词规则，但必须能加载和识别模型提供的词汇表和分词规则，输出一致的 token ID 序列即可。**
- 处理用户输入时：
  - 分词器将文本 → token → 编号 ID
  - 模型用 token ID 索引 **Embedding 矩阵（词汇表的嵌入矩阵）**
  - 获得该 token 的语义向量（Embedding 向量）
  - 后续进入 Transformer 层计算

**分词器类型：**

| 类型 | 算法 | 代表模型 | 特点 |
| ----- | ----- | ----- | ----- |
| **BPE** (Byte Pair Encoding) | 合并最频繁字节对 | GPT-2, RoBERTa, Llama | 子词级别，从字符开始合并 |
| **WordPiece** | 基于概率最大化 | BERT, DistilBERT | 选择提升似然最大的子词对 |
| **SentencePiece** | 统一编码（BPE/Unigram） | T5, Llama-2, ALBERT | 将空格视为特殊字符，语言无关 |
| **Unigram** | 语言模型概率 | XLNet, SentencePiece 可选 | 从大量候选逐步剪枝 |
| **Bytes-level BPE** | 字节级 BPE | GPT-2, BLOOM | 直接处理字节，无需<unk> |


**AutoTokenizer 属于哪种类型？**

- AutoTokenizer 不是具体的分词器类型，而是一个 **工厂类/自动选择器**，它能自动根据模型选择对应的分词器类型（BPE/WordPiece/SentencePiece等），本身不是具体的分词算法。
- Qwen3.5 的分词算法位于 `/home/godev/backup/llm-aiops/lib/python3.12/site-packages/transformers/models/qwen3_5/tokenization_qwen3_5.py`，其中 `/home/godev/backup/llm-aiops/lib/python3.12/site-packages/` 是模块安装路径，其中分词器类是 **Qwen3_5Tokenizer**。
- Kimi2.6 的分词算法位于下载的模型词汇表所在的目录中，如 `$HOME/.cache/huggingface/hub/models--moonshotai--Kimi-K2.6/snapshots/7eb5002f6aadc958aed6a9177b7ed26bb94011bb/tokenization_kimi.py` 中的分词器类是 **TikTokenTokenizer**。
- 因此，分词算法所在的分词器类可以在 transformers 模块中，也可以在下载的模型词汇表目录中。

| AutoTokenizer 加载的模型 | 实际分词器类（以实际定义为准）         | 底层类型                 |
| ----------------------- | ----------------------------------- | ----------------------- |
| BERT                    | BertTokenizer                       | WordPiece               |
| GPT-2                   | GPT2Tokenizer                       | Bytes-level BPE         |
| GPT-4 / ChatGPT         | TiktokenTokenizer / GPT2Tokenizer   | BPE                     |
| Llama                   | LlamaTokenizer                      | SentencePiece (BPE)     |
| T5                      | T5Tokenizer                         | SentencePiece (Unigram) |
| Qwen                    | QwenTokenizer                       | SentencePiece / BPE     |

In [73]:
!pip install transformers

Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 841.2 kB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 873.6 kB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 806.8 kB/s eta 0:00:000:00:01m eta 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2


In [76]:
import sys, transformers

print(f"{'当前执行路径：':3s} {sys.executable}")
print(f"{'transformers 版本：':3s} {transformers.__version__}")
print(f"{'transformers 执行文件：':3s} {transformers.__file__}")

当前执行路径： /home/godev/backup/llm-aiops/bin/python3
transformers 版本： 4.36.0
transformers 执行文件： /home/godev/backup/llm-aiops/lib/python3.12/site-packages/transformers/__init__.py


In [77]:
# 加载 AutoTokenizer 类
# 语法：from 包/模块 import 类/函数/变量
from transformers import AutoTokenizer

In [88]:
# 加载 moonshotai/Kimi-K2.6 模型中的词汇表与分词规则
# 注意：使用 trust_remote_code=True 参数取消交互式信息
tokenizer_k26 = AutoTokenizer.from_pretrained('moonshotai/Kimi-K2.6', trust_remote_code=True)
tokenizer_q3 = AutoTokenizer.from_pretrained('Qwen/Qwen3-8B', trust_remote_code=True)
#type(tokenizer)

In [95]:
# 返回 token ID（处理过程如下所示）

text = "今天天气如何呢？"

token_k26 = tokenizer_k26.encode(text)
token_q3 = tokenizer_q3.encode(text)
print(f"Kimi-K26 token ID 映射：{token_k26}")
print(f"Qwen3-8B token ID 映射：{token_q3}")

Kimi-K26 token ID 映射：[3108, 8597, 93208, 864]
Qwen3-8B token ID 映射：[100644, 104307, 100007, 101036, 11319]


In [92]:
# 返回原始文本信息
text_k26 = tokenizer_k26.decode(token_k26)
text_q3 = tokenizer_q3.decode(token_q3)
print(f"Kimi-K26 的原始序列：{text_k26}")
print(f"Qwen3-8B 的原始序列：{text_q3}")

Kimi-K26 的原始序列：今天天气如何呢？
Qwen3-8B 的原始序列：今天天气如何呢？


In [96]:
# 不同分词器验证：加载相同模型可获得相同的 token ID
from transformers import AutoTokenizer
import tiktoken

text = "今天天气如何呢？"
tokenizer_hf = AutoTokenizer.from_pretrained('gpt2')
print(f"HuggingFace 分词器的 token ID 映射：{tokenizer_hf.encode(text)}")         # 第三方分词器
tokenizer_tiktoken = tiktoken.get_encoding('gpt2')
print(f"OpenAI 系列分词器的 token ID 映射：{tokenizer_tiktoken.encode(text)}")    # 官方发布的分词器

HuggingFace 分词器的 token ID 映射：[20015, 232, 25465, 25465, 36365, 242, 36685, 224, 19526, 243, 37772, 95, 171, 120, 253]
OpenAI 系列分词器的 token ID 映射：[20015, 232, 25465, 25465, 36365, 242, 36685, 224, 19526, 243, 37772, 95, 171, 120, 253]


> 1. **tiktoken 分词器**仅支持 OpenAI 模型不支持第三方开源模型，如 GPT-2 (gpt2), GPT-3 (p50k_base, p50k_edit, r50k_base), GPT-3.5/4 (cl100k_base), GPT-4o/o1/o3 (o200k_base)。</br>
> 2. **transformers 的 AutoTokenizer 类**可支持第三方开源模型，如 Qwen, Kimi, DeepSeek 等。

In [32]:
# 场景1：如何根据 token ID 找到其在 Qwen3 词汇表中对应的编码信息（UTF-8 字节编码）？
from pathlib import Path
from transformers import AutoTokenizer
import sys, json

vocab_file = "/home/godev/.cache/huggingface/hub/models--Qwen--Qwen3-8B/snapshots/b968826d9c46dd6066d109eabc6255188de91218/tokenizer.json"
# 定义模型词汇表路径
text = "你今天觉得如何？"

if Path(vocab_file).exists():
    try:
        print(f"模型词汇表路径：{vocab_file}\n")
    except ZeroDivisionErrors as e:
        print(f"错误：{e}")
        sys.exit(1)
# 判断模型词汇表文件是否存在，不存在退出程序。

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-8B')

for id in tokenizer.encode(text):
    with open(vocab_file, 'r', encoding='utf-8', errors='replace') as f:
        data = json.load(f)    # 加载 JSON 文件为字典结构
        
    vocab_hash = data['model']['vocab']    # 索引模型词汇表
    #print(vocab)    # 尽量不要在 Jupyter 中输出词汇表字典防止出现以下告警！
    swapped = { v: k for k, v in vocab_hash.items() }    # 字典推导式：将编码与 token ID 的映射转换为 token ID 与编码的映射（键值位置互换）
    print(f"模型词汇表 token ID：{id:7d} 映射编码 {swapped[id]}")

模型词汇表路径：/home/godev/.cache/huggingface/hub/models--Qwen--Qwen3-8B/snapshots/b968826d9c46dd6066d109eabc6255188de91218/tokenizer.json

模型词汇表 token ID：  56568 映射编码 ä½ł
模型词汇表 token ID： 100644 映射编码 ä»Ĭå¤©
模型词汇表 token ID：  99801 映射编码 è§īå¾Ĺ
模型词汇表 token ID： 100007 映射编码 å¦Ĥä½ķ
模型词汇表 token ID：  11319 映射编码 ï¼Ł


> 注意：Jupyter 输出速率限制触发以下报错，不是代码本身的问题。在短时间内打印大量数据时，Jupyter 为了保护前端浏览器不被卡死，会自动暂停输出。

```plaintext
IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)
```

In [28]:
# 场景2：如何根据 token ID 找到其在 Kimi-K2.6 词汇表中对应的编码信息（base64 编码）？
from pathlib import Path
from transformers import AutoTokenizer
import sys, re, base64

vocab_file = "/home/godev/.cache/huggingface/hub/models--moonshotai--Kimi-K2.6/snapshots/7eb5002f6aadc958aed6a9177b7ed26bb94011bb/tiktoken.model"
text = "如何实现文本分词？"

tokenizer = AutoTokenizer.from_pretrained('moonshotai/Kimi-K2.6', trust_remote_code=True)
# 未加 trust_remote_code=True 参数将交互式提示是否信任模型中提供的运行脚本，输入 y 继续运行。
token = tokenizer.encode(text)

if Path(vocab_file).exists():
    try:
        print(f"模型词汇表路径：{vocab_file}\n")
    except ZeroDivisionErrors as e:
        print(f"错误：{e}")
        sys.exit(1)

for id in token:
    id_str = str(id)    # 字符串转换：re 模块正则表达式只能识别字符串
    with open(vocab_file, 'r', encoding='utf-8') as f:
        #pattern = re.compile(rf'^[a-zA-Z0-9+/-\=]+ {re.escape(id_str)}$')
        # 正则表达式-方法1：re.compile() 先编译正则，返回对象，再进行匹配加速运行。
        for line in f:
            # 迭代词汇表中每行进行正则匹配
            #matches = pattern.search(line.strip())
            matches = re.findall(rf'^[a-zA-Z0-9+/-\=]+ {re.escape(id_str)}$', line.strip())
            # 正则表达式-方法2：正则中的变量引用需使用格式化 re.escape() 方法
            if matches:
                '''
                Kimi-K2.6 词汇表的编码过程：原始文本 > UTF-8 字节 > base64 编码
                解码过程：base64 编码（当前状态）> UTF-8 字节 > 原始文本
                '''
                print(f"原始 token ID 映射：{matches}")             # matches 为单个元素的列表对象
                b64_seq = matches[0].split(' ')[0]                 # 取出 base64 编码
                seq = base64.b64decode(b64_seq).decode('utf-8')    # 解编码 UTF-8 字节为原始文本
                print(f"token ID 映射关系：{id_str} → {b64_seq} → {seq}\n-----")

模型词汇表路径：/home/godev/.cache/huggingface/hub/models--moonshotai--Kimi-K2.6/snapshots/7eb5002f6aadc958aed6a9177b7ed26bb94011bb/tiktoken.model

原始 token ID 映射：['5aaC5L2V5a6e546w 128129']
token ID 映射关系：128129 → 5aaC5L2V5a6e546w → 如何实现
-----
原始 token ID 映射：['5paH5pys 26386']
token ID 映射关系：26386 → 5paH5pys → 文本
-----
原始 token ID 映射：['5YiG 671']
token ID 映射关系：671 → 5YiG → 分
-----
原始 token ID 映射：['6K+N 3961']
token ID 映射关系：3961 → 6K+N → 词
-----
原始 token ID 映射：['77yf 864']
token ID 映射关系：864 → 77yf → ？
-----


**待解决：**
1. BPE 是什么？
2. 构建 embedding 层完成 token 的嵌入与语义矩阵的构建？

In [1]:
# 查看指定分词器所在的原始路径
import inspect
from transformers import Qwen3_5Tokenizer

print(inspect.getfile(Qwen3_5Tokenizer))

/home/godev/backup/llm-aiops/lib/python3.12/site-packages/transformers/models/qwen3_5/tokenization_qwen3_5.py
